# SIFLOW · run_5 · Dream-7B setup (SIFLOW-D)

Downloads Dream-7B, verifies a masked forward, and tokenizes data in **Dream's own tokenizer**. Default path trains live in run_6 (Dream-7B ~14GB fits on A100-80GB). An optional cell precomputes a reduced-support cache if you prefer a cache->train split.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_0 passed

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

> **Note:** if `tokenizer.mask_token_id` is unset for Dream, set `teacher.mask_token` in `siflow/config/dream.yaml` (the model card documents the mask token). If a raw forward doesn't expose `.logits`, set `teacher.auto_class` to the documented class.

In [ ]:
# Verify the teacher loads and a masked forward is sensible
import torch
from siflow.teacher import DreamTeacher
teacher = DreamTeacher(dtype=torch.bfloat16)
print("vocab", teacher.vocab_size, "hidden", teacher.hidden_dim, "mask_id", teacher.mask_index)
ids = torch.full((1, 16), teacher.mask_index, device=teacher.device)
z = teacher.logits(ids)
print("argmax tokens:", z.argmax(-1)[0][:8].tolist())

In [ ]:
# Tokenize data in Dream's tokenizer (train + val)
from siflow.data import build_token_chunks
tokz = teacher.tokenizer
n  = build_token_chunks(tokz, 256, 60_000, f"{BASE}/data/dream_256.npy",
                        dataset="Skylion007/openwebtext", split="train")
nv = build_token_chunks(tokz, 256, 2_000, f"{BASE}/data/dream_val.npy",
                        dataset="Skylion007/openwebtext", split="train", skip_seqs=60_000)
print("dream chunks:", n, nv)

In [ ]:
# (OPTIONAL) Precompute a reduced-support cache instead of training live.
# Resumable at shard granularity. Skip this cell to train live in run_6.
# !python scripts/build_cache.py --config siflow/config/dream.yaml \
#     --tokens {BASE}/data/dream_256.npy --out {BASE}/cache/dream --n 50000 --m 128 --batch 8

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---

print('saved to', BASE)